# VAYU Climate Digital Twin - Kaggle GPU Training (Central India) **1981-2025**

**Accelerator**: GPU T4 x2 (recommended) or T4 x1 / P100
**Dataset to attach**: `shyam31415/vayu-central-india-1981-2025`

## What changed from the 2010-2025 runs

**45 years instead of 16.** IMD rainfall/tmax/tmin now span 1981-2025, and every
auxiliary source was extended to match, so training windows go from ~4,350 to
~14,900 (stride 1).

**All 17 input channels are populated for the first time.** Previously
`insat_lst`, `insat_sst` and `chirps_rain` were constant zero in every region,
and `uwnd_850`/`vwnd_850`/`shum_850` were additionally dead in Indo-Gangetic
Plain and Central India - meaning those two regions had no moisture or
circulation predictor at all. Now:

| channel | source | note |
|---|---|---|
| `insat_lst` | ERA5-Land skin temperature | **substitute** for INSAT-3D LST (MOSDAC never approved) |
| `insat_sst` | NOAA OISST v2.1 | **substitute** for INSAT-3D SST (MOSDAC never approved) |
| `uwnd_850`, `vwnd_850`, `shum_850` | NCEP/NCAR R1, India subset | server-side subset, 1981-2025 |
| `chirps_rain` | CHIRPS v2.0 | auxiliary predictor; IMD stays the target |

`shum_850` is the strongest non-climatology predictor of next-day rainfall in
every region - it was the dead channel that mattered most.

## Measured baselines BEFORE training (the numbers to beat)

Best fixed persistence/climatology blend, leads 1-7 pooled, train 1981-2021 /
val 2022 (`scripts/skill_ceiling_probe.py`):

| | R2_rain | R2_tmax | R2_tmin |
|---|---|---|---|
| **Central India** | **+0.263** | **+0.879** | **+0.905** |

**Read this carefully:** the untrained blend already clears the nominal
R2_rain >= 0.20 / R2_tmax >= 0.80 targets in most regions. Hitting those
thresholds therefore proves nothing - a day-of-year lookup table plus
yesterday's observation gets there. Success is a **positive skill score over
the blend**.

`scripts/linear_headroom_probe.py` additionally measured, at lead-1, that
**temperature is saturated** (a ridge regression on all live channels beats the
blend by at most +0.004, and by -0.001 in one region) while **rainfall has real
headroom** (+0.027 to +0.099). This run therefore prioritises rainfall
(`--rain-weight 2.5`) and treats "do not regress below the temperature
blend" as the temperature goal.

## Literature context
- Narula et al., [arXiv:2402.07851](https://arxiv.org/abs/2402.07851) (NeurIPS 2025 CCAI): Autoformers on IMD 0.25 deg 1901-2023 beat ECMWF HRES by ~22% lower error at 1 day, ~27% at 3 days. They report **per-lead JJAS relative error vs NWP**, not pooled all-year R2.
- Ghosh et al., [arXiv:2607.26581](https://arxiv.org/abs/2607.26581) (Jul 2026): across ten methods on daily Indian rainfall grids, ConvLSTM did **not** consistently beat simpler baselines, and **persistence had the best high-rainfall detection in all four cities tested**. Beating persistence here is a real result, not a low bar.
- IMD 0.25 deg rainfall dataset: cite **Pai et al. (2014), MAUSAM 65(1), pp. 1-18**.

## Steps
1. Settings -> Accelerator -> **GPU T4 x2**
2. Add Input -> `shyam31415/vayu-central-india-1981-2025`
3. Run all cells top to bottom

Train stride is 6 for this region (scaled by node count so the session
finishes); 1981-2021 is 14,975 days, so the old stride 3 would not complete.


In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('Dependencies installed')

In [ ]:
# ── Mount project code and locate dataset ───────────────────────────────────────────────
import sys, os
from pathlib import Path

REGION = 'central_india'
REPO_DIR = '/kaggle/working/isro'
PROCESSED_DIR = f'{REPO_DIR}/data/processed_{REGION}'
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/{REGION}_main'

# IMPORTANT: verify this repo has today's changes before relying on this clone.
if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'rm -rf {REPO_DIR}')
    os.system(f'git clone https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

# Filenames for the 1981-2025 rebuild (the 1981-2025 names are gone).
NORMALIZED_FILE = 'normalized_1981-2025.nc'
NORM_PARAMS_FILE = 'norm_params_1981-2025.nc'

# NOTE: must run after clone/rm -rf above, since these dirs are nested inside
# REPO_DIR and would otherwise be wiped out by rm -rf.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Validation lives in the staging cell below, which locates every file
# individually. That matters because a bundle is often split across sibling
# folders (…-1-001 / …-1-002) and Kaggle may store a colliding name as
# `train_sequences-001.pt`. Requiring exact names here caused a false failure.
# Under --all-windows the *_sequences.pt files are not needed at all, since
# windows are sliced from normalized_*.nc.
root = Path('/kaggle/input')
_seen = sorted({p.name for p in root.rglob('*') if p.is_file()})
print(f'{len(_seen)} file(s) visible under /kaggle/input:')
for _n in _seen:
    print('   ', _n)


In [ ]:
# -- Stage bundle files (resilient to multi-folder Kaggle datasets) --------
# Files are located individually rather than from one DATASET_DIR, because a
# bundle can be split across sibling folders (...-1-001 / ...-1-002). Copying
# from a single folder silently skipped files in earlier runs.
#
# NOTE the 1981-2025 bundles intentionally contain NO sequence_manifest.json
# and NO *_sequences.pt. Windows are sliced lazily from normalized_1981-2025.nc
# via --all-windows, so the pre-built tensors are dead weight. Requiring
# sequence_manifest.json here (as the 2010-2025 notebooks did) aborts the run
# before training with "Missing required files".
import shutil
from pathlib import Path as _P

_ROOT = _P('/kaggle/input')

REQUIRED_FILES = [
    NORM_PARAMS_FILE,
    NORMALIZED_FILE,
    'elevation.nc',
    'lsm.nc',
]
OPTIONAL_FILES = ['pipeline_log_1981-2025.json', 'static_raster_manifest.json']


def _locate(name):
    """Find `name` anywhere under /kaggle/input, preferring this region's bundle."""
    matches = sorted(_ROOT.rglob(name))
    if not matches:
        stem, dot, ext = name.rpartition('.')
        if dot:
            matches = sorted(_ROOT.rglob(f'{stem}-[0-9][0-9][0-9].{ext}'))
        if matches:
            print(f'note: {name} not found; using suffixed variant {matches[0].name}')
    if not matches:
        return None
    preferred = [m for m in matches if 'central_india' in str(m).replace('-', '_')]
    return (preferred or matches)[0]


_missing = []
for _f in REQUIRED_FILES + OPTIONAL_FILES:
    _src = _locate(_f)
    if _src is None:
        if _f in REQUIRED_FILES:
            _missing.append(_f)
        else:
            print(f'optional, not found: {_f}')
        continue
    shutil.copy(_src, _P(PROCESSED_DIR) / _f)
    print(f'staged {_f:32s} <- {_src.parent.name}/{_src.name}')

if _missing:
    raise RuntimeError(
        'Missing required files: ' + ', '.join(_missing) +
        '. Attach the complete central_india 1981-2025 dataset via "Add Input".'
    )

os.system(f'ls -lah {PROCESSED_DIR}')


In [ ]:
# -- Smoke check: verify model and data before full training ---------------
# Rewritten for the 1981-2025 bundles. The old check loaded
# sequence_manifest.json and train_sequences.pt to read the feature count;
# neither exists any more, so the channel count is read straight from the
# normalized dataset and the smoke run uses --all-windows at a wide stride.
import subprocess, sys
import xarray as _xr
PY = sys.executable

_ds = _xr.open_dataset(f'{PROCESSED_DIR}/{NORMALIZED_FILE}')
print('time steps :', _ds.sizes['time'])
print('grid       :', _ds.sizes['lat'], 'x', _ds.sizes['lon'])
print('data_vars  :', list(_ds.data_vars))
if _ds.sizes['time'] < 16000:
    raise RuntimeError(
        f"expected ~16436 daily steps for 1981-2025, got {_ds.sizes['time']} - "
        "stale dataset attached?"
    )
_ds.close()
print('OK: 1981-2025 normalized dataset present')

_r = subprocess.run([PY, '-m', 'ai_engine.trainer',
    '--data-dir',        PROCESSED_DIR,
    '--checkpoint-dir',  f'{REPO_DIR}/checkpoints/{REGION}_smoke',
    '--epochs',          '1',
    '--device',          'auto',
    '--smoke-only',
    '--normalized-file', f'{PROCESSED_DIR}/{NORMALIZED_FILE}',
    '--elevation-file',  f'{PROCESSED_DIR}/elevation.nc',
    '--lsm-file',        f'{PROCESSED_DIR}/lsm.nc',
    '--all-windows',
    '--train-start-year', '1981', '--train-end-year', '2021',
    '--val-start-year',   '2022', '--val-end-year',   '2022',
    '--test-start-year',  '2023', '--test-end-year',  '2025',
    '--train-stride',     '60',
    '--eval-stride',      '60'],
    cwd=REPO_DIR, capture_output=True, text=True)
print(_r.stdout[-3000:] if _r.stdout else '')
if _r.returncode != 0:
    print('\n=== smoke STDERR ===')
    print(_r.stderr[-4000:] if _r.stderr else '(empty)')
    raise RuntimeError(f'Smoke check failed (exit {_r.returncode})')
print('\nSmoke check PASSED - model + lazy windows + loss all wired correctly')


In [ ]:
# -- FINAL training run (1981-2025, 45 years) ------------------------------
# Measured floors for this region BEFORE training, from
# scripts/skill_ceiling_probe.py (best fixed persistence/climatology blend,
# leads 1-7 pooled, train 1981-2021 / val 2022):
#
#     R2_rain +0.263    R2_tmax +0.879    R2_tmin +0.905
#
# These are the numbers to BEAT. The blend alone already meets this project's
# nominal R2_rain >= 0.20 / R2_tmax >= 0.80 targets in most regions, so hitting
# those thresholds is not evidence of a working model - only a positive skill
# score over the blend is.
#
# --rain-weight 2.5: rainfall is the ONLY target with measured headroom
# (linear_headroom_probe: rain +0.027..+0.099 vs temperature +/-0.004).
# --rain-heavy-alpha 0.0: plain weighted MSE, exactly aligned with the R2 being
# reported. A second run at alpha 3.0 would trade R2 for extreme-event scores.
# --train-stride 6: 1981-2021 is 14,975 days (3.4x the old 2010-2021 record),
# so stride 3 would not finish inside a Kaggle session.
import os, subprocess, sys
PY = sys.executable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([PY, '-m', 'ai_engine.trainer',
    '--data-dir',               PROCESSED_DIR,
    '--checkpoint-dir',         CHECKPOINT_DIR,
    '--epochs',                 '25',
    '--device',                 'auto',
    '--amp',
    '--batch-size',             '1',
    '--grad-accum-steps',       '8',
    '--cosine-lr',
    '--early-stopping-patience','8',
    '--weight-decay',           '1e-4',
    '--gnn-dropout',            '0.12',
    '--lambda-conservation',    '0.0',
    '--lambda-smoothness',      '0.0',
    '--rain-weight',            '2.5',
    '--rain-heavy-alpha',       '0.0',
    '--norm-params-file',       f'{PROCESSED_DIR}/{NORM_PARAMS_FILE}',
    '--normalized-file',        f'{PROCESSED_DIR}/{NORMALIZED_FILE}',
    '--elevation-file',         f'{PROCESSED_DIR}/elevation.nc',
    '--lsm-file',               f'{PROCESSED_DIR}/lsm.nc',
    '--all-windows',
    '--train-start-year',       '1981',
    '--train-end-year',         '2021',
    '--val-start-year',         '2022',
    '--val-end-year',           '2022',
    '--test-start-year',        '2023',
    '--test-end-year',          '2025',
    '--train-stride',           '6',
    '--eval-stride',            '6',
    '--run-baselines',
    '--require-benchmarks'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {CHECKPOINT_DIR}')


In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / 'training_history.json'
if not history_path.exists():
    print('No training_history.json yet — run the training cell first.')
else:
    history = json.loads(history_path.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
    axes[0].plot(history['epochs'], history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Central India Training Loss')
    axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['epochs'], history['val_r2'], color='green', label='R2 Tmax')
    axes[1].axhline(0.80, color='red', linestyle='--', label='Target R2=0.80')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('R2'); axes[1].set_title('Validation R2')
    axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves_central_india.png', dpi=150)
    plt.show()
    print('Best val_loss:', min(history['val_loss']))
    if history['benchmark_metrics']:
        last = history['benchmark_metrics'][-1]
        print(f"Latest validation R2_tmax={last.get('r2_tmax'):.3f} | R2_tmin={last.get('r2_tmin'):.3f} | R2_rain={last.get('r2_rain'):.3f}")

In [ ]:
# -- Held-out test results (2023-2025, never seen in training/validation) --
# Read against the MEASURED blend floors, not against the nominal thresholds.
import json
from pathlib import Path

FLOORS = {'rainfall': 0.263, 'temp_max': 0.879, 'temp_min': 0.905}

test_report_path = Path(CHECKPOINT_DIR) / 'test_report.json'
if not test_report_path.exists():
    print('No test_report.json - check the training cell output for errors.')
else:
    test_results = json.loads(test_report_path.read_text())
    print(f"{'variable':10s} {'R2':>8s} {'floor':>8s} {'vs floor':>9s} "
          f"{'skill_clim':>11s}  verdict")
    for var, metrics in test_results.items():
        if not isinstance(metrics, dict) or 'r2' not in metrics:
            continue
        floor = FLOORS.get(var)
        r2 = metrics['r2']
        sk = metrics.get('skill_vs_climatology', float('nan'))
        if floor is None:
            print(f'{var:10s} {r2:>+8.3f}')
            continue
        delta = r2 - floor
        verdict = 'BEATS blend' if delta > 0.005 else (
            'matches blend' if delta > -0.005 else 'BELOW blend')
        print(f'{var:10s} {r2:>+8.3f} {floor:>+8.3f} {delta:>+9.3f} '
              f'{sk:>+11.3f}  {verdict}')

    print()
    print('Per-lead / JJAS / extreme-event metrics are in the same report under')
    print('the verification block - read those for literature comparison')
    print('(Narula et al. arXiv:2402.07851 report per-lead JJAS relative error,')
    print('not pooled all-year R2).')


In [ ]:
import shutil
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    dst = f'/kaggle/working/vayu_best_{REGION}.pt'
    shutil.copy(best_ckpt, dst)
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint ready: {dst} ({size_mb:.1f} MB)')
else:
    print('vayu_best.pt not found — check training cell output for errors.')